# Ingestión del archivo `movie_company.json`

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-16")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
dbutils.widgets.text("p_environment", "")
v_environment = dbutils.widgets.get("p_environment")

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

## 1. Leer el archivo JSON usando `DataFrameReader` de Spark

In [0]:
movie_company_schema = "movieId INT, companyId INT"

movie_company_df = (spark.read 
    .schema(movie_company_schema)
    .option("multiLine", True)
    .csv(f"{bronze_folder_path}/{v_file_date}/movie_company")
)
display(movie_company_df)

movieId,companyId
51942,41680
51995,6667
52010,79522
52010,79523
52010,9096
52015,1360
52015,2341
52015,3768
52032,8828
52067,2361


## 2. Cambiar el nombre de las columnas según lo requerido

In [0]:
movie_company_renamed_df = (movie_company_df
    .withColumnRenamed("movieId", "movie_id")
    .withColumnRenamed("companyId", "company_id")
)

## 3. Agregar las columnas `ingestion_date` y `environmate` al DateFrame

In [0]:
from pyspark.sql.functions import current_timestamp, lit

movie_company_final_df = add_ingestion_date(movie_company_renamed_df).withColumn("enviroment", lit(v_environment)).withColumn("file_date", lit(v_file_date))


## 4. Escribir datos en el datalake en formato `Parquet`

In [0]:
merge_delta_lake( movie_company_final_df, "movie_silver", "movies_companies", "tgt.movie_id = src.movie_id AND tgt.company_id = src.company_id AND tgt.file_date = src.file_date", "file_date" )

In [0]:
%sql
SELECT * FROM movie_silver.movies_companies

path,name,size,modificationTime
abfss://silver@moviehistory4.dfs.core.windows.net/movies_companies/_SUCCESS,_SUCCESS,0,1789061842000
abfss://silver@moviehistory4.dfs.core.windows.net/movies_companies/_committed_4474307030048210501,_committed_4474307030048210501,424,1789061841000
abfss://silver@moviehistory4.dfs.core.windows.net/movies_companies/_started_4474307030048210501,_started_4474307030048210501,0,1789061839000
abfss://silver@moviehistory4.dfs.core.windows.net/movies_companies/part-00000-tid-4474307030048210501-9c99a9e4-56a0-48e8-868f-012c197b40b9-249-1-c000.snappy.parquet,part-00000-tid-4474307030048210501-9c99a9e4-56a0-48e8-868f-012c197b40b9-249-1-c000.snappy.parquet,26226,1789061840000
abfss://silver@moviehistory4.dfs.core.windows.net/movies_companies/part-00001-tid-4474307030048210501-9c99a9e4-56a0-48e8-868f-012c197b40b9-250-1-c000.snappy.parquet,part-00001-tid-4474307030048210501-9c99a9e4-56a0-48e8-868f-012c197b40b9-250-1-c000.snappy.parquet,25416,1789061840000
abfss://silver@moviehistory4.dfs.core.windows.net/movies_companies/part-00002-tid-4474307030048210501-9c99a9e4-56a0-48e8-868f-012c197b40b9-251-1-c000.snappy.parquet,part-00002-tid-4474307030048210501-9c99a9e4-56a0-48e8-868f-012c197b40b9-251-1-c000.snappy.parquet,24702,1789061840000
abfss://silver@moviehistory4.dfs.core.windows.net/movies_companies/part-00003-tid-4474307030048210501-9c99a9e4-56a0-48e8-868f-012c197b40b9-252-1-c000.snappy.parquet,part-00003-tid-4474307030048210501-9c99a9e4-56a0-48e8-868f-012c197b40b9-252-1-c000.snappy.parquet,13312,1789061840000
